In [1]:
%%html
<style>
div.jp-OutputArea-output pre {
    white-space: pre;
}
</style>
%run ./html.ipynb

# Initialize spark session

In [2]:
import pyspark


print(pyspark.__version__)

3.5.0


In [3]:
import os


os.environ['PYSPARK_SUBMIT_ARGS'] = "--packages org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.11.0,org.apache.iceberg:iceberg-aws-bundle:1.11.0,org.projectnessie.nessie-integrations:nessie-spark-extensions-3.5_2.12:0.105.3 pyspark-shell"

In [4]:
spark.stop()

NameError: name 'spark' is not defined

In [4]:
from pyspark.sql import SparkSession


spark = (
    SparkSession.builder
    .master("local[*]")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.defaultCatalog", "iceberg")
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.iceberg.type", "rest")
    .config("spark.sql.catalog.iceberg.uri", "http://nessie.nessie.svc.cluster.local:19120/iceberg")
    # .config("spark.sql.catalog.iceberg.warehouse", "iceberg")
    .getOrCreate()
)

def run(query: str):
    spark.sql(query).show(truncate=False)

# Query

In [5]:
run("CREATE NAMESPACE IF NOT EXISTS sandbox")

++
||
++
++



In [6]:
run("SHOW CATALOGS")
run("SHOW SCHEMAS")
run("SHOW TABLES FROM sandbox")
run("SHOW VIEWS FROM sandbox")

+-------------+
|catalog      |
+-------------+
|iceberg      |
|spark_catalog|
+-------------+

+---------+
|namespace|
+---------+
|sandbox  |
+---------+

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+

+---------+--------+-----------+
|namespace|viewName|isTemporary|
+---------+--------+-----------+
+---------+--------+-----------+



In [7]:
run("""
        CREATE TABLE IF NOT EXISTS sandbox.employee (
            id INT,
            role STRING,
            department STRING,
            salary FLOAT,
            region STRING)
        USING iceberg
    """)

++
||
++
++



In [8]:
run("SHOW TABLES FROM sandbox")

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|sandbox  |employee |false      |
+---------+---------+-----------+



In [9]:
run("SELECT * FROM sandbox.employee")

+---+----+----------+------+------+
|id |role|department|salary|region|
+---+----+----------+------+------+
+---+----+----------+------+------+



In [10]:
run("SHOW TBLPROPERTIES sandbox.employee")

+-------------------------------+----------------------------------------------------------------+
|key                            |value                                                           |
+-------------------------------+----------------------------------------------------------------+
|created-at                     |2026-07-06T07:14:29.433326483Z                                  |
|current-snapshot-id            |none                                                            |
|format                         |iceberg/parquet                                                 |
|format-version                 |2                                                               |
|gc.enabled                     |false                                                           |
|nessie.catalog.content-id      |31cda7f8-9191-4562-be67-e22c7b2ef98e                            |
|nessie.commit.id               |5d38f3c92de337b6346f586f788b927586284846793be5b88fa794d98513d548|
|nessie.co